# ColQwen3 — Multimodal Late-Interaction Retrieval

ColQwen3 enables **text-to-image** and **text-to-text** retrieval
using Qwen3-VL with ColPali-style multi-vector pooling.

Multimodal late-interaction retrieval via vLLM.

In [ ]:
import os
import sys
import time

import numpy as np
import torch

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "plugins"))
sys.path.insert(0, os.path.join(ROOT, "models"))

## Load ColQwen3

In [ ]:
from vllm import LLM

llm = LLM(
    model="VAGOsolutions/SauerkrautLM-ColQwen3-1.7b-Turbo-v0.1",
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    skip_mm_profiling=True,
    mm_processor_cache_gb=1,
    limit_mm_per_prompt={"image": 1},
)
print("✅ ColQwen3 loaded")

## Encode Text Queries

In [ ]:
queries = [
    "What is the architecture of transformer models?",
    "How does attention mechanism work in deep learning?",
    "Explain retrieval-augmented generation.",
    "What are the benefits of multi-vector retrieval?",
]

print(f"Encoding {len(queries)} queries...")
t0 = time.perf_counter()
outputs = llm.encode(queries, pooling_task="token_embed")
elapsed = time.perf_counter() - t0

for i, o in enumerate(outputs):
    emb = o.outputs.data
    if hasattr(emb, 'cpu'):
        emb = emb.cpu().numpy()
    else:
        emb = np.array(emb)
    if emb.ndim == 1:
        dim = 128
        emb = emb.reshape(-1, dim)
    print(f"  Query {i}: {emb.shape[0]} token vectors × {emb.shape[1]} dim")

print(f"\n✅ Encoded in {elapsed:.2f}s ({len(queries)/elapsed:.0f} queries/s)")

## Throughput Benchmark

In [ ]:
text = "The quick brown fox jumps over the lazy dog. " * 20

for bs in [1, 8]:
    batch = [text] * bs
    for _ in range(5):
        llm.encode(batch, pooling_task="token_embed")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        llm.encode(batch, pooling_task="token_embed")
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / 10
    print(f"Batch {bs:3d}: {bs/elapsed:.1f} req/s  ({elapsed*1000:.1f}ms)")

## Summary

ColQwen3 via vLLM Factory:
- Full multimodal ColPali pipeline via vLLM
- Multimodal: text + image inputs via ColPali pooling
- cos_sim = 0.9997 (query), 0.9900 (image) parity vs colpali-engine